# TrackNetV3 — Fine-tuning con PadelTracker100

Pipeline completo para entrenar un detector de pelota específico para pádel.

## Orden de ejecución

**Primera vez (Drive vacío):**
1. Celda A — Setup (GPU + Drive + variables)
2. Celda B — Descargar dataset desde Zenodo (~15 min)
3. Celda C — Preparar TrackNetV3 (clonar + parches + pesos)
4. Celda D — Extraer frames (~40 min)
5. Celda E — Convertir anotaciones a CSV
6. Celda F — Crear splits val/ y test/
7. Celda G — **Training** (~3-4h)
8. Celda H — Inferencia sobre tu vídeo

**Sesiones siguientes (datos ya en Drive):**
1. Celda A — Setup
2. Celda C — Preparar TrackNetV3
3. Celda G — Training (retoma desde el último checkpoint)

⚠️ Todo se guarda en Drive — los checkpoints sobreviven desconexiones.

In [ ]:
# ============================================================
# CELDA A — Setup: GPU + Drive + variables globales
# Ejecuta SIEMPRE en cada sesión nueva
# ============================================================
import subprocess, os

# Verificar GPU
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else '❌ SIN GPU — activa T4 en Runtime > Change runtime type')

# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Variables globales — usadas en todas las celdas
DRIVE_ROOT        = '/content/drive/MyDrive/paddelstats_tracknet'
DATASET_DIR       = f'{DRIVE_ROOT}/padeltracker100'
LABELS_DIR        = f'{DATASET_DIR}/labels'
TRACKNET_DATA_DIR = f'{DRIVE_ROOT}/tracknetv3_dataset'
TRACKNET_DIR      = f'{DRIVE_ROOT}/TrackNetV3'
CKPT_DIR          = f'{DRIVE_ROOT}/ckpts'
EXP_DIR           = f'{DRIVE_ROOT}/exp_padel'
TARGET_W, TARGET_H = 960, 540

MATCH_CONFIG = [
    ('match1', '2022_BCN_FinalF_1', f'{LABELS_DIR}/2022_BCN_FinalF_1_ball.json'),
    ('match2', '2022_BCN_FinalM_1', f'{LABELS_DIR}/2022_BCN_FinalM_1_ball.json'),
]

for d in [DRIVE_ROOT, DATASET_DIR, LABELS_DIR, TRACKNET_DATA_DIR, CKPT_DIR, EXP_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Variables y directorios listos')

In [ ]:
# ============================================================
# CELDA B — Descargar PadelTracker100 desde Zenodo
# Solo necesaria si los mp4s no están en Drive
# DOI: 10.5281/zenodo.14653706 — padel-data-labels.zip (7.1 GB)
# ============================================================
import os

mp4_f = f'{DATASET_DIR}/2022_BCN_FinalF_1.mp4'
if os.path.exists(mp4_f) and os.path.getsize(mp4_f) > 1e8:
    print('✅ Dataset ya descargado, saltando')
else:
    ZIP_PATH = f'{DATASET_DIR}/padel-data-labels.zip'
    ZIP_URL  = 'https://zenodo.org/records/14653706/files/padel-data-labels.zip?download=1'

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1e8:
        print('Descargando padel-data-labels.zip (~7.1 GB)...')
        !wget -q --show-progress -O "{ZIP_PATH}" "{ZIP_URL}"

    print('Extrayendo...')
    !unzip -q "{ZIP_PATH}" -d "{DATASET_DIR}"

    import glob, shutil
    for mp4 in glob.glob(f'{DATASET_DIR}/**/*.mp4', recursive=True):
        dst = os.path.join(DATASET_DIR, os.path.basename(mp4))
        if mp4 != dst: shutil.move(mp4, dst)
    for j in glob.glob(f'{DATASET_DIR}/**/*.json', recursive=True):
        dst = os.path.join(LABELS_DIR, os.path.basename(j))
        if j != dst: shutil.move(j, dst)

    print('✅ Extracción completada')

# Verificar
for f in ['2022_BCN_FinalF_1.mp4', '2022_BCN_FinalM_1.mp4',
          'labels/2022_BCN_FinalF_1_ball.json', 'labels/2022_BCN_FinalM_1_ball.json']:
    path = os.path.join(DATASET_DIR, f)
    ok = os.path.exists(path) and os.path.getsize(path) > 1000
    print(f'{'✅' if ok else '❌'} {f} ({os.path.getsize(path)/1e6:.0f} MB)' if os.path.exists(path) else f'❌ {f} — no encontrado')

In [ ]:
# ============================================================
# CELDA C — Preparar TrackNetV3
# Ejecuta SIEMPRE en cada sesión nueva (parches + pesos + checkpoint)
# ============================================================
import os, shutil, torch, gdown
from pathlib import Path

!pip install parse --quiet

# 1. Clonar TrackNetV3
if not os.path.exists(f'{TRACKNET_DIR}/train.py'):
    !git clone https://github.com/qaz812345/TrackNetV3.git "{TRACKNET_DIR}" --quiet
    print('✅ TrackNetV3 clonado')
else:
    print('✅ TrackNetV3 ya existe')

# 2. Parche dataset.py — siempre se aplica para garantizar rutas correctas
dataset_py = f'{TRACKNET_DIR}/dataset.py'
with open(dataset_py) as f:
    src = f.read()

# Reemplazar data_dir sea cual sea su valor actual
import re
src = re.sub(r"data_dir = '.*?'", f"data_dir = '{TRACKNET_DATA_DIR}'", src)
src = src.replace(
    "w, h = Image.open(os.path.join(rally_dir, f'0.{IMG_FORMAT}')).size",
    "first = sorted([x for x in os.listdir(rally_dir) if x.endswith(IMG_FORMAT)])[0]; w, h = Image.open(os.path.join(rally_dir, first)).size"
)
src = src.replace(
    "f_file = np.array([os.path.join(rally_dir, f'{f_id}.{IMG_FORMAT}') for f_id in label_df['Frame']])",
    "f_file = np.array([os.path.join(rally_dir, f'{int(f_id):06d}.{IMG_FORMAT}') for f_id in label_df['Frame']])"
)
with open(dataset_py, 'w') as f:
    f.write(src)
print(f'✅ dataset.py parcheado → data_dir = {TRACKNET_DATA_DIR}')

# 3. Parche utils/general.py — IMG_FORMAT png → jpg
general_py = f'{TRACKNET_DIR}/utils/general.py'
with open(general_py) as f:
    src = f.read()
src = re.sub(r"IMG_FORMAT = '.*?'", "IMG_FORMAT = 'jpg'", src)
with open(general_py, 'w') as f:
    f.write(src)
print('✅ utils/general.py parcheado → IMG_FORMAT = jpg')

# 4. Descargar pesos pre-entrenados si no existen
tracknet_pt = f'{CKPT_DIR}/TrackNet_best.pt'
if not os.path.exists(tracknet_pt):
    print('Descargando pesos pre-entrenados (bádminton)...')
    ckpt_zip = f'{CKPT_DIR}/ckpts.zip'
    gdown.download(id='1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA', output=ckpt_zip, quiet=False)
    !unzip -q "{ckpt_zip}" -d "{CKPT_DIR}"
    for pt in Path(CKPT_DIR).rglob('*.pt'):
        dst = f'{CKPT_DIR}/{pt.name}'
        if str(pt) != dst: shutil.move(str(pt), dst)
    print('✅ Pesos descargados')
else:
    print('✅ Pesos ya descargados')

for pt_name in ['TrackNet_best.pt', 'InpaintNet_best.pt']:
    src_pt, dst_pt = f'{CKPT_DIR}/{pt_name}', f'{EXP_DIR}/{pt_name}'
    if os.path.exists(src_pt) and not os.path.exists(dst_pt):
        shutil.copy(src_pt, dst_pt)

# 5. Preparar TrackNet_cur.pt — siempre verificar que existe y tiene las claves correctas
cur_pt = f'{EXP_DIR}/TrackNet_cur.pt'
if not os.path.exists(cur_pt):
    shutil.copy(f'{EXP_DIR}/TrackNet_best.pt', cur_pt)
    print('✅ TrackNet_cur.pt creado')

ckpt = torch.load(cur_pt, map_location='cpu')
updated = False
for k, v in {'mask_ratio': 0, 'max_val_acc': 0}.items():
    if k not in ckpt.get('param_dict', {}):
        ckpt.setdefault('param_dict', {})[k] = v; updated = True
if 'max_val_acc' not in ckpt:
    ckpt['max_val_acc'] = 0; updated = True
if updated:
    torch.save(ckpt, cur_pt)
print('✅ Checkpoint listo')

# 6. Instalar dependencias
!pip install -r "{TRACKNET_DIR}/requirements.txt" --quiet 2>&1 | tail -2
print('\n✅ Celda C completada — TrackNetV3 listo')

In [ ]:
# ============================================================
# CELDA D — Extraer frames de los mp4s
# Solo necesaria si los frames no están en Drive (~40 min)
# Reanuda automáticamente si se cortó
# ============================================================
import cv2, os
from pathlib import Path
from tqdm import tqdm

VIDEOS = [
    ('match1', '2022_BCN_FinalF_1', f'{DATASET_DIR}/2022_BCN_FinalF_1.mp4'),
    ('match2', '2022_BCN_FinalM_1', f'{DATASET_DIR}/2022_BCN_FinalM_1.mp4'),
]

for match_name, video_name, video_path in VIDEOS:
    out_dir = f'{TRACKNET_DATA_DIR}/train/{match_name}/frame/{video_name}'
    os.makedirs(out_dir, exist_ok=True)

    existing = sorted(Path(out_dir).glob('*.jpg'))
    last_idx = int(existing[-1].stem) if existing else -1

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if last_idx >= total - 10:
        cap.release()
        print(f'✅ {match_name}: {len(existing)} frames ya extraídos')
        continue

    print(f'Extrayendo {match_name} ({total} frames, ya hay {last_idx+1})...')
    frame_idx = written = 0
    with tqdm(total=total, initial=last_idx+1) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: break
            if frame_idx > last_idx:
                cv2.imwrite(f'{out_dir}/{frame_idx:06d}.jpg',
                            cv2.resize(frame, (TARGET_W, TARGET_H)),
                            [cv2.IMWRITE_JPEG_QUALITY, 90])
                written += 1
                if written % 500 == 0: os.sync()
            frame_idx += 1
            pbar.update(1)
    cap.release()
    os.sync()
    print(f'✅ {match_name}: {len(list(Path(out_dir).glob("*.jpg")))} frames extraídos')

In [ ]:
# ============================================================
# CELDA E — Convertir anotaciones COCO JSON → CSV TrackNetV3
# Solo necesaria si los CSVs no están en Drive
# ============================================================
import json, os, time, shutil
import pandas as pd
from pathlib import Path

ORIG_W, ORIG_H   = 1920, 1080
BALL_CATEGORY_ID = 1

def convert_ball_json(json_path, output_csv):
    with open(json_path) as f:
        data = json.load(f)
    img_map = {}
    for img in data.get('images', []):
        stem = Path(img['file_name']).stem
        frame_num = int(''.join(filter(str.isdigit, stem)))
        img_map[img['id']] = {'frame': frame_num,
                               'orig_w': img.get('width', ORIG_W),
                               'orig_h': img.get('height', ORIG_H)}
    ball_anns = {}
    for ann in data.get('annotations', []):
        if ann.get('category_id') != BALL_CATEGORY_ID: continue
        img_id = ann['image_id']
        bbox = ann.get('bbox')
        if bbox and img_id in img_map:
            info = img_map[img_id]
            sx, sy = TARGET_W / info['orig_w'], TARGET_H / info['orig_h']
            ball_anns[img_id] = (round((bbox[0] + bbox[2]/2) * sx),
                                  round((bbox[1] + bbox[3]/2) * sy))
    rows = []
    for img_id, info in sorted(img_map.items(), key=lambda x: x[1]['frame']):
        if img_id in ball_anns:
            cx, cy = ball_anns[img_id]
            rows.append({'Frame': info['frame'], 'Visibility': 1, 'X': cx, 'Y': cy})
        else:
            rows.append({'Frame': info['frame'], 'Visibility': 0, 'X': 0, 'Y': 0})
    df = pd.DataFrame(rows)
    tmp = f'/tmp/{Path(output_csv).name}'
    df.to_csv(tmp, index=False)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    shutil.copy2(tmp, output_csv)
    os.sync()
    return len(df), (df['Visibility'] == 1).sum()

for match_name, video_name, json_path in MATCH_CONFIG:
    csv_dir = f'{TRACKNET_DATA_DIR}/train/{match_name}/csv'
    out_csv = f'{csv_dir}/{video_name}_ball.csv'
    if os.path.exists(out_csv) and os.path.getsize(out_csv) > 1000:
        print(f'✅ {match_name}: CSV ya existe')
        continue
    n_frames, n_vis = convert_ball_json(json_path, out_csv)
    time.sleep(1)
    ok = os.path.exists(out_csv) and os.path.getsize(out_csv) > 1000
    print(f'{'✅' if ok else '❌'} {match_name}: {n_frames} frames, {n_vis} con pelota ({n_vis/n_frames*100:.1f}%)')

print('\n✅ Celda E completada')

In [ ]:
# ============================================================
# CELDA F — Crear splits val/ y test/
# Solo necesaria si no existen en Drive
# ============================================================
import os, shutil, pandas as pd
from pathlib import Path

def create_split(split, src_match, src_video, n_frames):
    dst_frame_dir = f'{TRACKNET_DATA_DIR}/{split}/match1/frame/{src_video}'
    dst_csv_dir   = f'{TRACKNET_DATA_DIR}/{split}/match1/csv'

    if os.path.exists(dst_frame_dir) and len(list(Path(dst_frame_dir).glob('*.jpg'))) >= n_frames:
        print(f'✅ {split}/match1 ya existe')
        return

    os.makedirs(dst_frame_dir, exist_ok=True)
    os.makedirs(dst_csv_dir, exist_ok=True)

    src_frames = sorted(Path(f'{TRACKNET_DATA_DIR}/train/{src_match}/frame/{src_video}').glob('*.jpg'))[-n_frames:]
    print(f'Creando {split}/match1 ({len(src_frames)} frames)...')
    for f in src_frames:
        dst = Path(dst_frame_dir) / f.name
        if not dst.exists(): shutil.copy2(f, dst)

    existing = sorted([int(f.stem) for f in Path(dst_frame_dir).glob('*.jpg')])
    df = pd.read_csv(f'{TRACKNET_DATA_DIR}/train/{src_match}/csv/{src_video}_ball.csv')
    df[df['Frame'].isin(existing)].to_csv(f'{dst_csv_dir}/{src_video}_ball.csv', index=False)
    print(f'✅ {split}/match1 creado ({len(existing)} frames)')

create_split('val',  'match2', '2022_BCN_FinalM_1', 500)
create_split('test', 'match1', '2022_BCN_FinalF_1', 300)
print('\n✅ Celda F completada')

In [ ]:
# ============================================================
# CELDA G — Fine-tuning
# Ejecuta SIEMPRE después de la celda C
# Retoma automáticamente si la sesión se cortó
# ============================================================
import os
EPOCHS = 10

os.chdir(TRACKNET_DIR)
print(f'Training — {EPOCHS} epochs')
print(f'Checkpoints: {EXP_DIR}')
print()

!pip install parse --quiet

!python train.py \
    --model_name TrackNet \
    --epochs {EPOCHS} \
    --save_dir "{EXP_DIR}" \
    --resume_training \
    --verbose

In [ ]:
# ============================================================
# CELDA G2 — Ver progreso del entrenamiento
# Puedes ejecutarla en cualquier momento mientras entrena
# ============================================================
import pandas as pd
from pathlib import Path

pts = sorted(Path(EXP_DIR).glob('*.pt'))
print('Checkpoints:')
for pt in pts:
    print(f'  {pt.name} ({pt.stat().st_size/1e6:.1f} MB)')

logs = list(Path(EXP_DIR).rglob('*.csv'))
if logs:
    log = pd.read_csv(logs[0])
    print(f'\nEpochs completados: {len(log)}')
    print(log.to_string())
else:
    print('\nAún no hay log de entrenamiento')

In [ ]:
# ============================================================
# CELDA H — Inferencia sobre tu vídeo de pádel
# Ejecutar después de que el training termine
# ============================================================
import os
from google.colab import files

print('Sube tu vídeo de prueba (ej: test_60s.mp4):')
uploaded = files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]

PRED_DIR    = f'{DRIVE_ROOT}/predictions'
TRACKNET_FT = f'{EXP_DIR}/TrackNet_best.pt'
INPAINT_PT  = f'{CKPT_DIR}/InpaintNet_best.pt'
os.makedirs(PRED_DIR, exist_ok=True)
os.chdir(TRACKNET_DIR)

print('Corriendo inferencia...')
!pip install parse --quiet
!python predict.py \
    --video_file "{VIDEO_PATH}" \
    --tracknet_file "{TRACKNET_FT}" \
    --inpaintnet_file "{INPAINT_PT}" \
    --save_dir "{PRED_DIR}" \
    --output_video \
    --large_video

In [ ]:
# ============================================================
# CELDA I — Resultados + descarga del vídeo anotado
# ============================================================
import pandas as pd
from pathlib import Path
from google.colab import files

pred_csvs = list(Path(PRED_DIR).glob('*.csv'))
if pred_csvs:
    df = pd.read_csv(pred_csvs[0])
    visible  = df[df['Visibility'] == 1]
    det_rate = len(visible) / len(df) * 100

    print('=' * 50)
    print('RESULTADO FINAL')
    print('=' * 50)
    print(f'Frames procesados : {len(df)}')
    print(f'Pelota detectada  : {len(visible)}')
    print(f'Detection rate    : {det_rate:.1f}%')
    print()
    print('Comparativa:')
    print(f'  YOLOv8 single-frame (baseline) : ~20%')
    print(f'  WASB/TrackNetV2 sin fine-tuning :   0%')
    print(f'  TrackNetV3 fine-tuned (pádel)   : {det_rate:.1f}%  ← ESTE')
    print()
    if det_rate > 60:
        print('✅ Listo para integrar en el pipeline de PaddelStats')
    elif det_rate > 30:
        print('⚠️ Mejora real pero insuficiente — prueba EPOCHS=20 en celda G')
    else:
        print('❌ Poca mejora — revisa los logs de entrenamiento en celda G2')

pred_videos = list(Path(PRED_DIR).glob('*.mp4'))
if pred_videos:
    print(f'\nDescargando vídeo anotado: {pred_videos[0].name}')
    files.download(str(pred_videos[0]))